# IDL Exercise 3 — Sentiment Analysis (IMDB)
### Tasks: RNN · GRU · MLP (Global Avg Pooling) · MLP + Restricted Self-Attention
**Before running anything: Runtime → Change runtime type → Hardware accelerator → T4 GPU**

In [ ]:
# ── 0. GPU CHECK + GOOGLE DRIVE MOUNT + OUTPUT DIRS ───────────────────────
import torch, os
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT available!'}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT  = '/content/drive/MyDrive/IDL_Ex3_Output'
PLOTS_DIR  = os.path.join(DRIVE_OUT, 'plots')
MODELS_DIR = os.path.join(DRIVE_OUT, 'models')
TABLES_DIR = os.path.join(DRIVE_OUT, 'tables')
for d in [DRIVE_OUT, PLOTS_DIR, MODELS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"\nOutputs will be saved to:")
print(f"  Plots  → {PLOTS_DIR}")
print(f"  Models → {MODELS_DIR}")
print(f"  Tables → {TABLES_DIR}")


In [ ]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.nn.functional import pad
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re, os, urllib.request, zipfile
from tqdm import tqdm

## Step 1 — Upload Dataset
Run the cell below and upload **`IMDB Dataset.csv`** from your computer when prompted.

In [ ]:
# ── 2. UPLOAD IMDB DATASET.CSV ────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # select IMDB Dataset.csv
CSV_PATH = 'IMDB Dataset.csv'
print(f"CSV loaded: {CSV_PATH}")

## Step 2 — Download GloVe Embeddings
Downloads once (~862 MB). If the runtime is restarted but Drive is still mounted the zip is already there and extraction is skipped.

In [ ]:
# ── 3. GLOVE DOWNLOAD & LOAD (idempotent) ─────────────────────────────────
EMBEDDING_SIZE = 100
MAX_LENGTH     = 100

def download_and_load_glove(dim=100):
    cache_dir = '/root/.vector_cache'
    os.makedirs(cache_dir, exist_ok=True)
    glove_txt = f'{cache_dir}/glove.6B.{dim}d.txt'
    glove_zip = f'{cache_dir}/glove.6B.zip'
    if not os.path.exists(glove_txt):
        if not os.path.exists(glove_zip):
            print('Downloading GloVe 6B (862 MB) ...')
            urllib.request.urlretrieve(
                'https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip', glove_zip)
        print('Extracting ...')
        with zipfile.ZipFile(glove_zip, 'r') as z:
            z.extract(f'glove.6B.{dim}d.txt', cache_dir)
    print('Loading vectors into memory ...')
    w2v = {}
    with open(glove_txt, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=400_000):
            parts = line.rstrip().split(' ')
            w2v[parts[0]] = torch.tensor([float(x) for x in parts[1:]], dtype=torch.float32)
    print(f'Loaded {len(w2v):,} word vectors.')
    return w2v

if 'word2vec' not in dir():
    word2vec = download_and_load_glove(dim=EMBEDDING_SIZE)
else:
    print('GloVe already loaded, skipping.')

## Step 3 — Data Loading & Preprocessing
Equivalent to `loader.py`. No torchtext dependency.

In [ ]:
# ── 4. DATA LOADING (loader.py — no torchtext) ────────────────────────────
Train_size = 30_000

def review_clean(text):
    text = re.sub(r'[^A-Za-z]+', ' ', text)
    text = re.sub(r'https?:/\/\S+', ' ', text)
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def tokinize(s):
    return review_clean(s).lower().split()[:MAX_LENGTH]

_zero_vec = torch.zeros(EMBEDDING_SIZE)

def get_vecs_by_tokens(tokens):
    vecs = [word2vec.get(t, _zero_vec) for t in tokens]
    return torch.stack(vecs) if vecs else _zero_vec.unsqueeze(0)

def preprocess_review(s):
    tokens   = tokinize(s)
    embedded = get_vecs_by_tokens(tokens)                          # [len, 100]
    if embedded.shape[0] < MAX_LENGTH:
        embedded = pad(embedded, (0, 0, 0, MAX_LENGTH - embedded.shape[0]))
    return embedded.unsqueeze(0)                                   # [1, 100, 100]

def preprocess_label(label):
    # positive -> [1,0]  |  negative -> [0,1]
    return [0.0, 1.0] if label == 'negative' else [1.0, 0.0]

class ReviewDataset(Dataset):
    def __init__(self, reviews, labels):
        self.reviews = list(reviews)
        self.labels  = list(labels)
    def __len__(self):
        return len(self.reviews)
    def __getitem__(self, idx):
        return self.reviews[idx], self.labels[idx]

def collate_batch(batch):
    label_list, review_list, embed_list = [], [], []
    for review, label in batch:
        label_list.append(preprocess_label(label))
        review_list.append(tokinize(review))
        embed_list.append(preprocess_review(review).detach())
    labels     = torch.tensor(label_list, dtype=torch.float32).reshape(-1, 2)
    embeddings = torch.cat(embed_list)                             # [B, 100, 100]
    return labels.to(device), embeddings.to(device), review_list

def get_data_set(batch_size):
    data       = pd.read_csv(CSV_PATH)
    train_data = data[:Train_size]
    test_data  = data[Train_size:].reset_index(drop=True)
    train_dl   = DataLoader(ReviewDataset(train_data['review'], train_data['sentiment']),
                            batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    test_dl    = DataLoader(ReviewDataset(test_data['review'],  test_data['sentiment']),
                            batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    return train_dl, test_dl, MAX_LENGTH, EMBEDDING_SIZE

print('Data utilities ready.')

## Step 4 — Model Definitions
Exact code from `sentiment_start.py`.

In [ ]:
# ── 5. MODEL DEFINITIONS ──────────────────────────────────────────────────

# Global needed by ExLRestSelfAtten constructor; overridden per-experiment.
atten_size = 5
output_size = 2

class MatMul(nn.Module):
    def __init__(self, in_channels, out_channels, use_bias=True):
        super().__init__()
        self.matrix   = nn.Parameter(
            nn.init.xavier_normal_(torch.empty(in_channels, out_channels)), requires_grad=True)
        self.use_bias = use_bias
        if use_bias:
            self.bias = nn.Parameter(torch.zeros(1, 1, out_channels), requires_grad=True)
    def forward(self, x):
        x = torch.matmul(x, self.matrix)
        if self.use_bias:
            x = x + self.bias
        return x


class ExRNN(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.hidden_size   = hidden_size
        self.in2hidden     = nn.Linear(input_size + hidden_size, hidden_size)
        self.hidden2output = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, output_size))
    def name(self): return 'RNN'
    def forward(self, x, hidden_state):
        combined = torch.cat((x, hidden_state), dim=1)
        hidden   = torch.tanh(self.in2hidden(combined))
        output   = self.hidden2output(hidden)
        return output, hidden
    def init_hidden(self, bs):
        return torch.zeros(bs, self.hidden_size, device=next(self.parameters()).device)


class ExGRU(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.hidden_size     = hidden_size
        self.reset_gate      = nn.Linear(input_size + hidden_size, hidden_size)
        self.update_gate     = nn.Linear(input_size + hidden_size, hidden_size)
        self.candidate_layer = nn.Linear(input_size + hidden_size, hidden_size)
        self.hidden2output   = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, output_size))
    def name(self): return 'GRU'
    def forward(self, x, hidden_state):
        combined        = torch.cat((x, hidden_state), dim=1)
        r               = torch.sigmoid(self.reset_gate(combined))
        z               = torch.sigmoid(self.update_gate(combined))
        reset_hidden    = r * hidden_state
        h_tilde         = torch.tanh(self.candidate_layer(torch.cat((x, reset_hidden), dim=1)))
        hidden          = z * hidden_state + (1 - z) * h_tilde
        output          = self.hidden2output(hidden)
        return output, hidden
    def init_hidden(self, bs):
        return torch.zeros(bs, self.hidden_size, device=next(self.parameters()).device)


class ExMLP(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.ReLU    = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.layer1  = MatMul(input_size,   hidden_size)
        self.layer2  = MatMul(hidden_size,  hidden_size)
        self.layer3  = MatMul(hidden_size,  output_size)
    def name(self): return 'MLP'
    def forward(self, x):
        x = self.dropout(self.ReLU(self.layer1(x)))
        x = self.dropout(self.ReLU(self.layer2(x)))
        x = self.layer3(x)        # [B, num_words, 2] — raw logits per word
        return x


class ExLRestSelfAtten(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.hidden_size      = hidden_size
        self.atten_size       = atten_size          # reads global set before instantiation
        self.window_size      = 2 * self.atten_size + 1
        self.sqrt_hidden_size = np.sqrt(float(hidden_size))
        self.ReLU    = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.softmax = nn.Softmax(dim=2)
        # Token-wise projection
        self.layer1  = MatMul(input_size,  hidden_size)
        # Single-head attention matrices
        self.W_q     = MatMul(hidden_size, hidden_size, use_bias=False)
        self.W_k     = MatMul(hidden_size, hidden_size, use_bias=False)
        self.W_v     = MatMul(hidden_size, hidden_size, use_bias=False)
        # Learnable relative positional encoding: one vector per offset in the window
        self.pos_embedding = nn.Parameter(torch.zeros(1, 1, self.window_size, hidden_size))
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)
        # Post-attention classifier (same as Task 2 MLP)
        self.layer2  = MatMul(hidden_size, hidden_size)
        self.layer3  = MatMul(hidden_size, output_size)
    def name(self): return 'MLP_atten'
    def forward(self, x):
        x      = self.dropout(self.ReLU(self.layer1(x)))
        # Build local neighborhood tensor: roll+pad approach
        padded = pad(x, (0, 0, self.atten_size, self.atten_size, 0, 0))
        x_nei  = torch.stack(
            [torch.roll(padded, k, dims=1) for k in range(-self.atten_size, self.atten_size + 1)],
            dim=2)                                                      # [B, T+2a, W, H]
        x_nei  = x_nei[:, self.atten_size:-self.atten_size, :, :]      # [B, T, W, H]
        x_nei  = x_nei + self.pos_embedding                            # add positional encoding
        # Compute query, keys, values
        query        = self.W_q(x)                                     # [B, T, H]
        keys         = self.W_k(x_nei)                                 # [B, T, W, H]
        vals         = self.W_v(x_nei)                                 # [B, T, W, H]
        # Scaled dot-product attention scores
        atten_scores  = (query.unsqueeze(2) * keys).sum(dim=-1) / self.sqrt_hidden_size
        atten_weights = self.softmax(atten_scores)                     # [B, T, W]
        # Weighted sum of values
        context = (atten_weights.unsqueeze(-1) * vals).sum(dim=2)      # [B, T, H]
        x = x + context                                                # residual
        x = self.dropout(self.ReLU(self.layer2(x)))
        x = self.layer3(x)                                             # [B, T, 2]
        return x, atten_weights


print('Model definitions ready.')

In [ ]:
# ── 6. print_review ───────────────────────────────────────────────────────
def print_review(rev_text, sbs1, sbs2, lbl1, lbl2):
    """
    rev_text : list of word tokens
    sbs1     : per-word positive logits  [num_words]
    sbs2     : per-word negative logits  [num_words]
    lbl1,lbl2: true one-hot values  (positive=[1,0], negative=[0,1])
    """
    n_words = len(rev_text)
    n_print = min(n_words, 30)
    pos_logits = np.array(sbs1[:n_words])
    neg_logits = np.array(sbs2[:n_words])
    logits     = np.stack([pos_logits, neg_logits], axis=1)
    # Per-word softmax probabilities
    shifted    = logits - logits.max(axis=1, keepdims=True)
    exp_l      = np.exp(shifted)
    probs      = exp_l / exp_l.sum(axis=1, keepdims=True)
    # Final prediction: average logits, then softmax
    final_l    = logits.mean(axis=0)
    final_e    = np.exp(final_l - final_l.max())
    final_p    = final_e / final_e.sum()
    true_label = 'positive' if lbl1 > lbl2 else 'negative'
    pred_label = 'positive' if final_p[0] > final_p[1] else 'negative'
    print('\nReview preview:')
    print(' '.join(rev_text[:n_print]))
    print(f'\nTrue label      : {true_label}')
    print(f'Predicted label : {pred_label}')
    print(f'Final avg logits: pos={final_l[0]:.4f}  neg={final_l[1]:.4f}')
    print(f'Final probs     : pos={final_p[0]:.4f}  neg={final_p[1]:.4f}')
    table = pd.DataFrame({
        'word'    : rev_text[:n_print],
        'pos_logit': np.round(pos_logits[:n_print], 4),
        'neg_logit': np.round(neg_logits[:n_print], 4),
        'pos_prob' : np.round(probs[:n_print, 0],  4),
        'neg_prob' : np.round(probs[:n_print, 1],  4),
    })
    print('\nPer-word sub-prediction scores:')
    print(table.to_string(index=False))
    print('-' * 70)
    return pred_label, true_label

In [ ]:
# ── 7. TRAINING UTILITIES ─────────────────────────────────────────────────
import io
from contextlib import redirect_stdout

def run_experiment(ModelClass, hidden_size,
                   run_recurrent, use_RNN=True, atten_size_val=0,
                   num_epochs=10, learning_rate=1e-3,
                   test_interval=50, batch_size=32):
    """Train one model config. Returns (model, history, exp_name)."""
    global atten_size
    atten_size = atten_size_val

    train_dl, test_dl, num_words, input_size = get_data_set(batch_size)
    model    = ModelClass(input_size, output_size, hidden_size).to(device)
    exp_name = f'{model.name()}_hidden{hidden_size}'

    print(f'\n{"="*65}')
    print(f'  Experiment : {exp_name}')
    print(f'  Device     : {device}   Epochs: {num_epochs}   LR: {learning_rate}')
    print(f'{"="*65}')

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_loss_ema = 1.0;  test_loss_ema = 1.0
    train_acc_ema  = 0.0;  test_acc_ema  = 0.0
    history = dict(train_steps=[], train_loss=[], train_acc=[],
                   test_steps=[],  test_loss=[],  test_acc=[])
    test_iterator = iter(test_dl)
    global_step   = 0

    for epoch in range(num_epochs):
        itr = 0
        for labels, reviews, reviews_text in train_dl:
            itr += 1; global_step += 1
            if (itr + 1) % test_interval == 0:
                test_iter = True
                try:
                    labels, reviews, reviews_text = next(test_iterator)
                except StopIteration:
                    test_iterator = iter(test_dl)
                    labels, reviews, reviews_text = next(test_iterator)
            else:
                test_iter = False

            target_labels = torch.argmax(labels, dim=1) if labels.dim() == 2 else labels
            if run_recurrent:
                hidden_state = model.init_hidden(int(labels.shape[0]))
                for i in range(num_words):
                    output, hidden_state = model(reviews[:, i, :], hidden_state)
            else:
                if atten_size_val > 0:
                    sub_score, _ = model(reviews)
                else:
                    sub_score = model(reviews)
                mask   = (reviews.abs().sum(dim=-1) > 0).float()
                output = (sub_score * mask.unsqueeze(-1)).sum(dim=1) \
                         / mask.sum(dim=1).unsqueeze(-1)
                target_labels = torch.argmax(labels, dim=1) if labels.dim() == 2 else labels

            loss      = criterion(output, target_labels)
            batch_acc = (torch.argmax(output, 1) == target_labels).float().mean().item()

            if not test_iter:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                train_loss_ema = 0.9 * float(loss.detach()) + 0.1 * train_loss_ema
                train_acc_ema  = 0.9 * batch_acc           + 0.1 * train_acc_ema
                history['train_steps'].append(global_step)
                history['train_loss'].append(train_loss_ema)
                history['train_acc'].append(train_acc_ema)
            else:
                test_loss_ema = 0.8 * float(loss.detach()) + 0.2 * test_loss_ema
                test_acc_ema  = 0.8 * batch_acc           + 0.2 * test_acc_ema
                history['test_steps'].append(global_step)
                history['test_loss'].append(test_loss_ema)
                history['test_acc'].append(test_acc_ema)
                print(f'Ep[{epoch+1}/{num_epochs}] St[{itr+1}/{len(train_dl)}]  '
                      f'TrLoss={train_loss_ema:.4f}  TeLoss={test_loss_ema:.4f}  '
                      f'TrAcc={train_acc_ema:.4f}  TeAcc={test_acc_ema:.4f}')

    model_path = os.path.join(MODELS_DIR, f'{exp_name}_epochs{num_epochs}.pth')
    torch.save(model.state_dict(), model_path)
    print(f'\nModel weights saved → {model_path}')
    return model, history, exp_name


def plot_and_save(histories, labels, title, filename):
    """Plot train/test loss and accuracy curves for one or more experiments."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
    for i, (h, lbl) in enumerate(zip(histories, labels)):
        c = colors[i % len(colors)]
        ax1.plot(h['train_steps'], h['train_loss'], '-',  color=c, alpha=0.8, label=f'{lbl} train')
        ax1.plot(h['test_steps'],  h['test_loss'],  '--', color=c, alpha=0.8, label=f'{lbl} test')
        ax2.plot(h['train_steps'], h['train_acc'],  '-',  color=c, alpha=0.8, label=f'{lbl} train')
        ax2.plot(h['test_steps'],  h['test_acc'],   '--', color=c, alpha=0.8, label=f'{lbl} test')
    for ax, ylabel, ts in zip([ax1, ax2], ['EMA Loss', 'EMA Accuracy'], ['Loss', 'Accuracy']):
        ax.set_title(ts, fontsize=12); ax.set_xlabel('Training step')
        ax.set_ylabel(ylabel); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Plot saved → {path}')
    plt.show()


def run_inference_on_texts(model, texts, true_labels, model_uses_atten=False, save_filename=None):
    """
    Run texts through a token-wise model (MLP / MLP_atten) and print per-word scores.
    If save_filename is given, output is also written to TABLES_DIR/save_filename.
    """
    model.eval()
    all_output = []
    with torch.no_grad():
        for text, true_label in zip(texts, true_labels):
            embedded = preprocess_review(text).to(device)
            tokens   = tokinize(text)
            if model_uses_atten:
                sub_score, _ = model(embedded)
            else:
                sub_score = model(embedded)
            lbl  = preprocess_label(true_label)
            subs = sub_score.detach().cpu().numpy()
            buf  = io.StringIO()
            with redirect_stdout(buf):
                print_review(tokens, subs[0, :, 0], subs[0, :, 1], lbl[0], lbl[1])
            captured = buf.getvalue()
            print(captured, end='')
            all_output.append(captured)
    if save_filename:
        path = os.path.join(TABLES_DIR, save_filename)
        with open(path, 'w', encoding='utf-8') as f:
            f.write(''.join(all_output))
        print(f'Table saved → {path}')
    model.train()


def run_inference_recurrent(model, texts, true_labels, descriptions=None, save_filename=None):
    """
    Run texts through a recurrent model (RNN/GRU).
    Prints true vs predicted sentiment and confidence for each review.
    If save_filename is given, output is also written to TABLES_DIR/save_filename.
    """
    model.eval()
    all_output = []
    with torch.no_grad():
        for idx, (text, true_label) in enumerate(zip(texts, true_labels)):
            embedded = preprocess_review(text).to(device)
            tokens   = tokinize(text)
            lbl      = preprocess_label(true_label)
            true_str = 'positive' if lbl[0] > lbl[1] else 'negative'
            hidden   = model.init_hidden(1)
            for i in range(MAX_LENGTH):
                output, hidden = model(embedded[:, i, :], hidden)
            probs    = torch.softmax(output, dim=1).cpu().numpy()[0]
            pred_str = 'positive' if probs[0] > probs[1] else 'negative'
            correct  = 'CORRECT ✓' if pred_str == true_str else 'WRONG   ✗'
            desc     = f'  [{descriptions[idx]}]\n' if descriptions else ''
            line = (f'\n{desc}'
                    f'  Review : {" ".join(tokens[:25])} ...\n'
                    f'  True   : {true_str:<10}  Predicted: {pred_str:<10}  '
                    f'P(pos)={probs[0]:.4f}  P(neg)={probs[1]:.4f}  {correct}')
            print(line)
            all_output.append(line)
    if save_filename:
        path = os.path.join(TABLES_DIR, save_filename)
        with open(path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(all_output))
        print(f'Results saved → {path}')
    model.train()


print('Training utilities ready.')


In [ ]:
# ── OPTIONAL: LOAD SAVED MODELS FROM DRIVE ────────────────────────────────
# Run this cell INSTEAD of all training cells below if models are already
# saved on your Drive and you want to skip retraining.
# (Safe to skip this cell if you just ran the training cells above.)

def load_model_from_drive(ModelClass, hidden_size, num_epochs=10, atten_size_val=0):
    global atten_size
    atten_size = atten_size_val
    model    = ModelClass(EMBEDDING_SIZE, output_size, hidden_size).to(device)
    exp_name = f'{model.name()}_hidden{hidden_size}'
    path     = os.path.join(MODELS_DIR, f'{exp_name}_epochs{num_epochs}.pth')
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    print(f'Loaded → {path}')
    return model

rnn_model_64    = load_model_from_drive(ExRNN,            64)
rnn_model_128   = load_model_from_drive(ExRNN,           128)
gru_model_64    = load_model_from_drive(ExGRU,            64)
gru_model_128   = load_model_from_drive(ExGRU,           128)
mlp_model_64    = load_model_from_drive(ExMLP,            64)
mlp_model_128   = load_model_from_drive(ExMLP,           128)
atten_model_64  = load_model_from_drive(ExLRestSelfAtten, 64,  atten_size_val=5)
atten_model_128 = load_model_from_drive(ExLRestSelfAtten, 128, atten_size_val=5)
print('\nAll models loaded from Drive.')


---
## Task 1 — RNN vs GRU (two hidden sizes each)
Runs 4 experiments: RNN-64, RNN-128, GRU-64, GRU-128.
Each model is saved to Drive. Comparison plots are saved after each pair.

In [ ]:
# ── TASK 1a: RNN hidden=64 ────────────────────────────────────────────────
rnn_model_64, rnn_hist_64, _ = run_experiment(
    ModelClass=ExRNN, hidden_size=64,
    run_recurrent=True, num_epochs=10)

In [ ]:
# ── TASK 1b: RNN hidden=128 ───────────────────────────────────────────────
rnn_model_128, rnn_hist_128, _ = run_experiment(
    ModelClass=ExRNN, hidden_size=128,
    run_recurrent=True, num_epochs=10)

In [ ]:
# ── RNN comparison plot ───────────────────────────────────────────────────
plot_and_save(
    histories=[rnn_hist_64, rnn_hist_128],
    labels   =['RNN hidden=64', 'RNN hidden=128'],
    title    ='Task 1 — RNN: hidden size comparison (64 vs 128)',
    filename ='task1_RNN_hidden64_vs_hidden128.png')

In [ ]:
# ── TASK 1c: GRU hidden=64 ────────────────────────────────────────────────
gru_model_64, gru_hist_64, _ = run_experiment(
    ModelClass=ExGRU, hidden_size=64,
    run_recurrent=True, num_epochs=10)

In [ ]:
# ── TASK 1d: GRU hidden=128 ───────────────────────────────────────────────
gru_model_128, gru_hist_128, _ = run_experiment(
    ModelClass=ExGRU, hidden_size=128,
    run_recurrent=True, num_epochs=10)

In [ ]:
# ── GRU comparison plot ───────────────────────────────────────────────────
plot_and_save(
    histories=[gru_hist_64, gru_hist_128],
    labels   =['GRU hidden=64', 'GRU hidden=128'],
    title    ='Task 1 — GRU: hidden size comparison (64 vs 128)',
    filename ='task1_GRU_hidden64_vs_hidden128.png')

In [ ]:
# ── TASK 1: TEST REVIEWS — RNN vs GRU ────────────────────────────────────
# Reviews designed to stress-test long-term memory retention.
# An RNN may forget early sentiment after many neutral filler words;
# a GRU's update/reset gates should preserve it better.

task1_reviews = [
    # 1. Positive signal early, many neutral words follow → tests memory of early state
    ("i loved this movie the acting was wonderful and the story was moving "
     "the rest of the film follows a family through several ordinary scenes "
     "in a small town with conversations meals walks and quiet moments before the credits",
     'positive',
     'Positive start + neutral tail — does the model remember early positive signal?'),

    # 2. Negative signal early, many neutral words follow → same test on negative side
    ("i hated this movie the acting was awful and the story was painful "
     "the rest of the film follows a family through several ordinary scenes "
     "in a small town with conversations meals walks and quiet moments before the credits",
     'negative',
     'Negative start + neutral tail — does the model remember early negative signal?'),

    # 3. Many positive words, then a hard reversal at the end ("but ... boring empty disappointing")
    ("the cinematography was beautiful the actors were famous the music was elegant "
     "and the scenes looked expensive but the movie was ultimately boring empty and disappointing",
     'negative',
     'Positive buildup, negative reversal at the end — does the model update on late signal?'),

    # 4. Many negative words, then a hard reversal ("but ... enjoyable")
    ("the beginning was slow awkward and confusing and i expected to dislike the film "
     "but by the end it became emotional clever and surprisingly enjoyable",
     'positive',
     'Negative buildup, positive reversal at the end — does the model update on late signal?'),
]

texts  = [r[0] for r in task1_reviews]
labels = [r[1] for r in task1_reviews]
descs  = [r[2] for r in task1_reviews]

print(f'\n{"="*65}')
print('  Task 1 — RNN hidden=128 on test reviews')
print(f'{"="*65}')
run_inference_recurrent(rnn_model_128, texts, labels, descriptions=descs,
                        save_filename='task1_RNN_hidden128_test_reviews.txt')

print(f'\n{"="*65}')
print('  Task 1 — GRU hidden=128 on test reviews')
print(f'{"="*65}')
run_inference_recurrent(gru_model_128, texts, labels, descriptions=descs,
                        save_filename='task1_GRU_hidden128_test_reviews.txt')


In [ ]:
# ── RNN vs GRU best-hidden comparison ────────────────────────────────────
# Using hidden=128 for both (pick whichever performed best above)
plot_and_save(
    histories=[rnn_hist_128, gru_hist_128],
    labels   =['RNN hidden=128', 'GRU hidden=128'],
    title    ='Task 1 — RNN vs GRU (hidden=128)',
    filename ='task1_RNN_vs_GRU_hidden128.png')

---
## Task 2 — Token-wise MLP with Global Average Pooling
Each word is processed independently by a 3-layer MLP producing a 2-class logit vector.
Final prediction = average of logits over all real (non-padded) words.
*(Softmax is applied AFTER averaging — averaging raw logits preserves the linear structure; applying softmax before would compress scores into [0,1] and distort the mean.)*

In [ ]:
# ── TASK 2: MLP hidden=64 ─────────────────────────────────────────────────
mlp_model_64, mlp_hist_64, _ = run_experiment(
    ModelClass=ExMLP, hidden_size=64,
    run_recurrent=False, atten_size_val=0, num_epochs=10)

In [ ]:
# ── TASK 2: MLP hidden=128 ────────────────────────────────────────────────
mlp_model_128, mlp_hist_128, _ = run_experiment(
    ModelClass=ExMLP, hidden_size=128,
    run_recurrent=False, atten_size_val=0, num_epochs=10)

In [ ]:
# ── TASK 2 ERROR ANALYSIS ─────────────────────────────────────────────────
# true_label must be 'positive' or 'negative'.

error_analysis_reviews = [

    # TP: true=positive, model predicts positive ✓
    ("This movie was absolutely fantastic, a masterpiece of cinema with brilliant acting "
     "and a gripping story that kept me on the edge of my seat the entire time",
     'positive',
     'TP — clearly positive review, model should predict positive'),

    # TN: true=negative, model predicts negative ✓
    ("This movie was terrible and painfully boring, the acting was wooden "
     "and the plot made no sense whatsoever",
     'negative',
     'TN — clearly negative review, model should predict negative'),

    # FP: true=negative, model predicts POSITIVE (wrong) ✗
    # Sarcastic/ironic negative review loaded with the highest-scoring positive words:
    # "masterpiece"(+32), "gripping"(+27), "brilliant"(+36), "fantastic"(+36), "surprisingly"(+30).
    # These dominate the per-word average and flip the MLP to predict positive.
    ("A masterpiece of wasted potential, gripping in its absolute pointlessness, "
     "with brilliant performers and a fantastic score that somehow make this "
     "the most surprisingly hollow cinema experience of the year",
     'negative',
     'FP — negative review disguised with positive-scoring words '
     '(masterpiece +32, brilliant +36, gripping +27, fantastic +36, surprisingly +30)'),

    # FN: true=positive, model predicts NEGATIVE (wrong) ✗
    # Genuinely positive review (low expectations exceeded) but packed with the
    # strongest negative-scoring words: "terrible"(-50), "wooden"(-51), "boring"(-51),
    # "bad"(-27), "painfully"(-24). These overwhelm the positive meaning.
    ("Going in expecting something terrible and wooden, dreading yet another boringly "
     "predictable story, I was completely blown away — this was not bad at all "
     "and far exceeded my painfully low expectations",
     'positive',
     'FN — positive review loaded with negative-scoring words '
     '(terrible -50, wooden -51, boring -51, bad -27, painfully -24)'),
]

# Use the best MLP model (hidden=128)
print('=== MLP Error Analysis ===')
for text, true_label, scenario in error_analysis_reviews:
    print(f'\n>>> Scenario: {scenario}')
    run_inference_on_texts(mlp_model_128, [text], [true_label], model_uses_atten=False)

# Save all four tables to a single file
run_inference_on_texts(
    mlp_model_128,
    [r[0] for r in error_analysis_reviews],
    [r[1] for r in error_analysis_reviews],
    model_uses_atten=False,
    save_filename='task2_MLP_hidden128_error_analysis_TP_TN_FP_FN.txt')


### Task 2 — Error Analysis
Define 4 custom reviews below to achieve **TP, FP, TN, FN** scenarios.
The cell runs them through the trained MLP and prints per-word sub-prediction scores.

In [ ]:
# ── TASK 2 ERROR ANALYSIS ─────────────────────────────────────────────────
# true_label must be 'positive' or 'negative'.

error_analysis_reviews = [

    # TP: true=positive, model predicts positive ✓
    # Unambiguously positive language — easy for the model.
    ("This movie was absolutely fantastic, a masterpiece of cinema with brilliant acting "
     "and a gripping story that kept me on the edge of my seat the entire time",
     'positive',
     'TP — clearly positive review, model should predict positive'),

    # TN: true=negative, model predicts negative ✓
    # Unambiguously negative language — easy for the model.
    ("This movie was terrible and painfully boring, the acting was wooden "
     "and the plot made no sense whatsoever",
     'negative',
     'TN — clearly negative review, model should predict negative'),

    # FP: true=negative, model predicts POSITIVE (wrong) ✗
    # A genuinely negative (sarcastic/ironic) review deliberately loaded with the
    # highest positive-scoring words the model knows:
    #   "masterpiece"(+32), "gripping"(+27), "brilliant"(+36),
    #   "fantastic"(+36), "surprisingly"(+30), "cinema"(+2.8).
    # These dominate the per-word averages and flip the prediction despite the
    # review being clearly negative in meaning.
    ("A masterpiece of wasted potential, gripping in its absolute pointlessness, "
     "with brilliant performers and a fantastic score that somehow make this "
     "the most surprisingly hollow cinema experience of the year",
     'negative',
     'FP — negative review disguised with positive-scoring words '
     '(masterpiece +32, brilliant +36, gripping +27, fantastic +36, surprisingly +30)'),

    # FN: true=positive, model predicts NEGATIVE (wrong) ✗
    # A genuinely positive review (low expectations, film exceeded them) but loaded
    # with the strongest negative-scoring words the model knows:
    #   "terrible"(-50), "wooden"(-51), "boring"(-51), "bad"(-27), "painfully"(-24).
    # These overwhelm the positive meaning and flip the prediction to negative.
    ("Going in expecting something terrible and wooden, dreading yet another boringly "
     "predictable story, I was completely blown away — this was not bad at all "
     "and far exceeded my painfully low expectations",
     'positive',
     'FN — positive review loaded with negative-scoring words '
     '(terrible -50, wooden -51, boring -51, bad -27, painfully -24)'),
]

# Use the best MLP model (hidden=128)
print('=== MLP Error Analysis ===')
for text, true_label, scenario in error_analysis_reviews:
    print(f'\n>>> Scenario: {scenario}')
    run_inference_on_texts(mlp_model_128, [text], [true_label], model_uses_atten=False)


---
## Task 3 — MLP + Restricted Self-Attention (window=5)
Same classifier as Task 2 but each word is first contextualized by attending to its
5 nearest neighbours on each side. Learnable Q/K/V matrices + relative positional encoding.

In [ ]:
# ── TASK 3: MLP + Restricted Self-Attention, hidden=64, atten_size=5 ─────
atten_model_64, atten_hist_64, _ = run_experiment(
    ModelClass=ExLRestSelfAtten, hidden_size=64,
    run_recurrent=False, atten_size_val=5, num_epochs=10)

In [ ]:
# ── TASK 3: MLP + Restricted Self-Attention, hidden=128, atten_size=5 ────
atten_model_128, atten_hist_128, _ = run_experiment(
    ModelClass=ExLRestSelfAtten, hidden_size=128,
    run_recurrent=False, atten_size_val=5, num_epochs=10)

In [ ]:
plot_and_save(
    histories=[atten_hist_64, atten_hist_128],
    labels   =['MLP_atten hidden=64', 'MLP_atten hidden=128'],
    title    ='Task 3 — MLP + Restricted Self-Attention: hidden size comparison',
    filename ='task3_MLPatten_hidden64_vs_hidden128.png')

In [ ]:
# ── TASK 3 ERROR ANALYSIS ─────────────────────────────────────────────────
# Same four reviews as Task 2 — compare per-word scores side by side.
print('=== MLP + Attention Error Analysis ===')
for text, true_label, scenario in error_analysis_reviews:
    print(f'\n>>> Scenario: {scenario}')
    run_inference_on_texts(atten_model_128, [text], [true_label], model_uses_atten=True)

# Save all four tables to a single file
run_inference_on_texts(
    atten_model_128,
    [r[0] for r in error_analysis_reviews],
    [r[1] for r in error_analysis_reviews],
    model_uses_atten=True,
    save_filename='task3_MLPatten_hidden128_error_analysis_TP_TN_FP_FN.txt')


### Task 3 — Error Analysis (same reviews, now with attention model)
Compare per-word scores with Task 2 results. The attention model should handle
context-dependent words (e.g., negations) better.

In [ ]:
# ── TASK 3 ERROR ANALYSIS ─────────────────────────────────────────────────
print('=== MLP + Attention Error Analysis ===')
for text, true_label, scenario in error_analysis_reviews:
    print(f'\n>>> Scenario: {scenario}')
    run_inference_on_texts(atten_model_128, [text], [true_label], model_uses_atten=True)

### Task 3 — Custom context-dependent review
A review where a word's meaning changes depending on its neighbour — MLP should fail, MLP+Atten should succeed.

In [ ]:
# ── TASK 3: CONTEXT-DEPENDENT REVIEW ─────────────────────────────────────
# This review is genuinely POSITIVE in meaning but every negative word
# appears immediately after "not", e.g. "not boring", "not terrible", "not wooden".
#
# MLP (no context): sees "boring"(-51), "terrible"(-50), "wooden"(-51), "bad"(-27)
#   as isolated words → predicts NEGATIVE (wrong).
# MLP+Atten (window=5): "not" falls within the attention window of each negative word
#   → the model can learn to flip its local score → predicts POSITIVE (correct).
#
# This is the canonical case where local context is essential.

context_review = ("This was not a boring film and not a terrible one either, "
                  "the actors were not wooden and the script was not bad, "
                  "leaving me surprisingly satisfied rather than disappointed")
context_label  = 'positive'

print('--- MLP (no context) ---')
run_inference_on_texts(mlp_model_128, [context_review], [context_label],
                       model_uses_atten=False,
                       save_filename='task3_MLP_hidden128_context_review.txt')

print('\n--- MLP + Attention (with local context) ---')
run_inference_on_texts(atten_model_128, [context_review], [context_label],
                       model_uses_atten=True,
                       save_filename='task3_MLPatten_hidden128_context_review.txt')


In [ ]:
# ── TASK 4: MLP vs MLP+ATTEN — CONTEXT-DEPENDENT REVIEWS ─────────────────
# These 5 reviews all rely on local context (negation, contrast) to convey
# their true sentiment.  An MLP without context should systematically fail
# on the negation cases; the MLP+Atten model (window=5) has "not" / "hardly"
# within its attention window and should handle them better.

task4_reviews = [
    # (review text, true label, description)

    # 1. "not bad" — negative word negated → should be positive
    ("this movie was not bad and actually quite enjoyable the story flowed well "
     "and the performances were decent",
     'positive',
     '"not bad" + enjoyable — MLP may fire on "bad"; Atten sees "not bad" as positive'),

    # 2. "not boring" — negative word negated → should be positive
    ("the film was not boring at all it was surprisingly fun and engaging from start to finish",
     'positive',
     '"not boring" — MLP fires on "boring"; Atten can contextualize the negation'),

    # 3. Contrastive "but" — positive clause follows negative buildup → should be positive
    ("i expected a terrible movie but it was actually good and the ending was satisfying",
     'positive',
     'Contrast "but": initial "terrible" vs. final "good" — does late context win?'),

    # 4. "hardly good" / "not enjoyable" — positive words negated → should be negative
    ("this movie was hardly good and certainly not enjoyable a waste of time altogether",
     'negative',
     '"hardly good" / "not enjoyable" — MLP fires on "good"/"enjoyable"; Atten inverts'),

    # 5. "not worth watching" with concession — overall negative
    ("the movie was not worth watching despite the good actors and the decent cinematography",
     'negative',
     '"not worth watching" despite positive details — true sentiment is negative'),
]

texts4  = [r[0] for r in task4_reviews]
labels4 = [r[1] for r in task4_reviews]
descs4  = [r[2] for r in task4_reviews]

print(f'\n{"="*65}')
print('  Task 4 — MLP hidden=128 on context-dependent reviews')
print(f'{"="*65}')
for text, true_label, desc in zip(texts4, labels4, descs4):
    print(f'\n>>> {desc}')
    run_inference_on_texts(mlp_model_128, [text], [true_label], model_uses_atten=False)

# Save all MLP tables to one file
run_inference_on_texts(
    mlp_model_128, texts4, labels4,
    model_uses_atten=False,
    save_filename='task4_MLP_hidden128_context_reviews.txt')

print(f'\n{"="*65}')
print('  Task 4 — MLP+Atten hidden=128 on context-dependent reviews')
print(f'{"="*65}')
for text, true_label, desc in zip(texts4, labels4, descs4):
    print(f'\n>>> {desc}')
    run_inference_on_texts(atten_model_128, [text], [true_label], model_uses_atten=True)

# Save all MLP+Atten tables to one file
run_inference_on_texts(
    atten_model_128, texts4, labels4,
    model_uses_atten=True,
    save_filename='task4_MLPatten_hidden128_context_reviews.txt')


In [ ]:
# ── FINAL SUMMARY PLOT: all models, best hidden size ─────────────────────
plot_and_save(
    histories=[rnn_hist_128, gru_hist_128, mlp_hist_128, atten_hist_128],
    labels   =['RNN-128', 'GRU-128', 'MLP-128', 'MLP+Atten-128'],
    title    ='All models — train/test loss & accuracy (hidden=128)',
    filename ='summary_all_models_hidden128.png')

print('\nAll outputs saved to Google Drive:', DRIVE_OUT)